In [28]:
import torch
import ipywidgets as widgets
from IPython.display import display, Javascript
import numpy as np
import cv2
import time
import asyncio

class DOR:
    def __init__(self, shader_func, res=128, fps_limit=60):
        self.shader_func = shader_func
        self.res = res
        self.target_dt = 1.0 / fps_limit
        
        # State variables
        self.t = 0.0
        self.running = False
        self.paused = False
        self.task = None
        
        self.img_widget = widgets.Image(format='jpeg', width=res, height=res)
        self.fps_label = widgets.Label(value="Ready")
        
        btn_layout = widgets.Layout(width='30px', padding='0px')
        
        self.btn_play = widgets.Button(description='', icon='pause', layout=btn_layout)
        self.btn_reset = widgets.Button(description='', icon='refresh', layout=btn_layout)
        self.btn_stop = widgets.Button(description='', icon='stop', button_style='danger', layout=btn_layout)
        
        self.controls = widgets.HBox([self.btn_play, self.btn_reset, self.btn_stop])
        self.ui = widgets.VBox([self.img_widget, self.fps_label, self.controls])
        
        self.btn_play.on_click(self.toggle_pause)
        self.btn_reset.on_click(self.reset_time)
        self.btn_stop.on_click(self.stop_render_btn)
        
        display(self.ui)

    def toggle_pause(self, _):
        self.paused = not self.paused
        self.btn_play.icon = 'play' if self.paused else 'pause'

    def reset_time(self, _):
        self.t = 0.0
        if self.paused:
            self.render_frame()

    def stop_render_btn(self, _):
        self.stop()
        self.fps_label.value = "Stopped."

    def stop(self):
        self.running = False
        if self.task:
            self.task.cancel()
            self.task = None

    def render_frame(self):
        with torch.no_grad():
            img_tensor = self.shader_func(self.res, self.t)
            img_tensor = (img_tensor.clamp(0, 1) * 255).byte()
            img_np = img_tensor.cpu().numpy()
            
        if img_np.shape[-1] == 3: 
            img_np = cv2.cvtColor(img_np, cv2.COLOR_RGB2BGR)
        elif img_np.shape[-1] == 4: 
            img_np = cv2.cvtColor(img_np, cv2.COLOR_RGBA2BGR)

        _, enc_data = cv2.imencode('.jpg', img_np, [int(cv2.IMWRITE_JPEG_QUALITY), 100])
        self.img_widget.value = enc_data.tobytes()

    async def loop(self):
        self.running = True
        self.btn_stop.disabled = False
        
        try:
            while self.running:
                loop_start = time.time()
                
                if not self.paused:
                    self.render_frame()
                    self.t += self.target_dt 
                
                process_time = time.time() - loop_start
                if process_time > 0:
                    self.fps_label.value = f"FPS: {1.0/process_time:.1f} | t={self.t:.2f}"
                
                # Corrected sleep logic to actually respect FPS limit
                wait = self.target_dt - process_time
                await asyncio.sleep(max(wait, 0.001))
                
        except asyncio.CancelledError:
            pass
        except Exception as e:
            self.fps_label.value = f"Error: {e}"
        finally:
            self.running = False

    def start(self):
        if self.task: self.task.cancel()
        if self.running: return
        self.task = asyncio.create_task(self.loop())


In [29]:
def shader(res, t):
    x = torch.linspace(-1, 1, res)
    y = torch.linspace(-1, 1, res)
    Y, X = torch.meshgrid(y, x, indexing='xy')
    
    R = torch.sqrt(X**2 + Y**2)
    A = torch.atan2(Y, X)
    
    wave = torch.sin(10 * R - t * 4.0) + torch.sin(A * 12 + t)
    
    r = (torch.sin(wave + t) * 0.5 + 0.5) ** 2
    g = (torch.sin(wave + t + 2.0) * 0.5 + 0.5) ** 2
    b = (torch.sin(wave + t + 4.0) * 0.5 + 0.5) ** .1
    
    return torch.stack([r, g, b], dim=-1)

try:dor.stop()
except NameError: pass
dor = DOR(shader, res=256, fps_limit=60)
dor.start()
